# GitHub MCP Server를 Amazon Bedrock AgentCore Gateway에 연결하기 - [URL 모드 Elicitation](https://modelcontextprotocol.io/specification/2025-11-25/client/elicitation#url-mode-flow)

Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved. SPDX-License-Identifier: Apache-2.0

Authorization Code Grant 유형을 지원하기 위해 두 가지 대상 생성 방법을 제공합니다. 

1. MCP Server 대상 생성 중 암시적 동기화

    이 방법에서는 관리자 사용자가 [CreateGatewayTarget](https://docs.aws.amazon.com/bedrock-agentcore-control/latest/APIReference/API_CreateGatewayTarget.html), [UpdateGatewayTarget](https://docs.aws.amazon.com/bedrock-agentcore-control/latest/APIReference/API_UpdateGatewayTarget.html) 또는 [SynchronizeGatewayTargets](https://docs.aws.amazon.com/bedrock-agentcore-control/latest/APIReference/API_SynchronizeGatewayTargets.html) 작업 중에 authorization code flow를 완료합니다. 이를 통해 AgentCore Gateway가 MCP server의 도구를 미리 검색하고 캐시할 수 있습니다.  

2. MCP Server 대상 생성 시 스키마를 미리 제공 

    이 방법에서는 AgentCore Gateway가 MCP server에서 도구 스키마를 동적으로 가져오는 대신, 관리자 사용자가 [CreateGatewayTarget](https://docs.aws.amazon.com/bedrock-agentcore-control/latest/APIReference/API_CreateGatewayTarget.html) 또는 [UpdateGatewayTarget](https://docs.aws.amazon.com/bedrock-agentcore-control/latest/APIReference/API_UpdateGatewayTarget.html) 작업 중에 도구 스키마를 직접 제공합니다. AgentCore Gateway는 제공된 스키마를 파싱하고 도구 정의를 캐시합니다. 따라서 관리자 사용자가 대상을 생성하거나 업데이트할 때 authorization code flow를 완료할 필요가 없습니다. Infrastructure as Code 파이프라인을 사용하여 AgentCore Gateway 리소스를 관리하는 경우처럼 생성/업데이트 작업 중 사람의 개입이 불가능할 때 권장되는 방식입니다. 또한 MCP server 대상에서 제공하는 모든 도구를 노출하지 않고 도구 스키마에 필요한 도구만 노출하려는 경우에도 유용합니다.  

    참고: 이 방법에서는 도구 스키마를 미리 제공하므로 [SynchronizeGatewayTargets](https://docs.aws.amazon.com/bedrock-agentcore-control/latest/APIReference/API_SynchronizeGatewayTargets.html) 작업을 지원하지 않습니다. 대상 구성을 업데이트하여 방법 1과 방법 2 사이에서 전환할 수 있습니다.      

즉, AgentCore Gateway 사용자는 캐시된 도구를 가져오는 `list/tools`를 호출할 때 MCP server 인증 서버의 인증을 요구받지 않습니다. Authorization code flow는 Gateway 사용자가 해당 MCP server의 도구를 호출할 때만 시작됩니다. 하나의 Gateway에 여러 MCP server가 연결된 경우 특히 유용합니다. 사용자는 모든 MCP server에서 인증하지 않고도 전체 도구 카탈로그(캐시된 도구)를 탐색할 수 있으며, 실제로 도구를 호출하는 특정 서버에 대해서만 flow를 완료하면 됩니다. 

### URL 세션 바인딩 

[URL 세션 바인딩](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)은 OAuth 권한 부여 요청을 시작한 사용자와 동의한 사용자가 동일한지 확인합니다. AgentCore Identity는 권한 부여 URL을 생성할 때 session-URI도 반환합니다. 사용자가 동의를 완료하면 브라우저는 session-URI와 함께 callback URL로 리디렉션됩니다. 이후 애플리케이션은 사용자의 ID와 session-URI를 모두 제시하여 [CompleteResourceTokenAuth](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/API_CompleteResourceTokenAuth.html) API를 호출해야 합니다. AgentCore Identity는 authorization code를 access token으로 교환하기 전에 flow를 시작한 사용자와 완료한 사용자가 동일한지 검증합니다. 이를 통해 사용자가 실수로 권한 부여 URL을 공유하여 다른 사람이 동의를 완료하고 잘못된 대상에게 access token이 발급되는 상황을 방지합니다. 권한 부여 URL과 session URI의 유효 시간은 10분으로 제한되어 오용 가능 시간을 더욱 줄입니다. 세션 바인딩은 관리자의 대상 생성(암시적 동기화)과 도구 호출 과정에 적용됩니다. 

### 보안 고려 사항

- **자격 증명 처리**: OAuth client secret은 `getpass`를 통해 수집되며 Notebook 실행 중에만 메모리에 저장됩니다. 프로덕션 환경에서는 secret을 AWS Secrets Manager에 저장하고 프로그래밍 방식으로 가져오세요.
- **최소 권한 IAM**: 이 Notebook에서 생성하는 AWS Identity and Access Management (IAM) 역할은 특정 AgentCore 리소스에 범위를 한정한 정책으로 최소 권한 원칙을 따릅니다.
- **토큰 만료**: Access token과 권한 부여 URL은 제한된 시간이 지나면 만료됩니다(일반적으로 token은 1시간, 권한 부여 URL은 10분). Refresh token을 사용할 수 있으면 AgentCore Identity가 만료된 token을 자동으로 갱신합니다.
- **로깅**: 프로덕션 배포에서는 AWS CloudTrail을 활성화하여 모든 AgentCore API 호출을 기록하고, Gateway 호출을 모니터링하도록 Amazon CloudWatch를 구성하세요.
- **공동 책임**: AWS는 AgentCore Gateway 인프라와 AgentCore Identity 서비스를 관리합니다. 고객은 OAuth 앱 자격 증명 보호, 적절한 IAM 정책 구성, 세션 바인딩을 위한 안전한 callback endpoint 구현을 책임집니다.

## GitHub [OAuth Apps](https://docs.github.com/en/apps/oauth-apps/using-oauth-apps) 설정 

In [ ]:
import getpass

GITHUB_CLIENT_ID = input("Enter your GitHub Client ID: ")
GITHUB_CLIENT_SECRET = getpass.getpass("Enter your GitHub Client Secret: ")

assert GITHUB_CLIENT_ID.strip(), "Client ID cannot be empty"
assert GITHUB_CLIENT_SECRET.strip(), "Client Secret cannot be empty"

In [ ]:
!pip install -r requirements.txt --force-reinstall -q

In [ ]:
import boto3
import os
from utils import utils
import json
import requests
import uuid

session = boto3.session.Session()
REGION = session.region_name

os.environ["AWS_DEFAULT_REGION"] = REGION
print(f"Using region: {REGION}")

GATEWAY_NAME = "ac-gateway-mcp-server"
CRED_PROVIDER_NAME = "github-mcp-server-provider"

agentcore_cp = boto3.client("bedrock-agentcore-control")
cognito = boto3.client("cognito-idp", region_name=REGION)

### 1단계: GitHub Credential Provider

In [ ]:
response = agentcore_cp.create_oauth2_credential_provider(
    name=CRED_PROVIDER_NAME,
    credentialProviderVendor="GithubOauth2",
    oauth2ProviderConfigInput={
        "githubOauth2ProviderConfig": {
            "clientId": GITHUB_CLIENT_ID,
            "clientSecret": GITHUB_CLIENT_SECRET,
        }
    },
    tags={"demo": "github-mcp-gateway"},
)

identity_callback = response["callbackUrl"]
cred_provider_arn = response["credentialProviderArn"]
secret_arn = response["clientSecretArn"]["secretArn"]

print(f"Callback URL: {identity_callback}")

**중요:** 생성한 [GitHub OAuth provider](https://github.com/settings/apps)로 돌아가 Authorization callback URL을 위 단계의 출력값으로 업데이트하세요.

In [ ]:
print("Go to https://github.com/settings/apps and update your GitHub App's")
print(f"Authorization callback URL to:\n\n  {identity_callback}\n")
input("Press Enter once you have updated the callback URL to continue...")

### 2단계: AgentCore Gateway 생성

### 2.1단계: Gateway의 인바운드 인증을 관리하도록 Amazon Cognito 설정

이 데모에서는 Amazon Cognito를 사용하여 AgentCore Gateway의 인바운드 인증을 관리합니다. 엔터프라이즈 요구 사항에 따라 OAuth 2.0을 준수하는 모든 IDP를 구성할 수 있습니다. 인바운드 구성은 AgentCore Gateway를 호출할 수 있는 사용자를 관리합니다. Okta, Entra ID, Auth 등의 [설정](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity-idp-microsoft.html) 단계를 확인하세요.

In [ ]:
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {
        "ScopeName": "invoke",  # 'invoke'만 입력하며 resource_server_id/invoke 형식으로 지정됩니다
        "ScopeDescription": "Scope for invoking the agentcore gateway",
    },
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)

In [ ]:
print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(
    cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES
)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(
    cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names
)

print(f"Client ID: {gw_client_id}")

# Discovery URL 가져오기
gw_cognito_discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration"
print(gw_cognito_discovery_url)

### 2.2단계: AgentCore Gateway AWS Identity and Access Management (IAM) 역할 생성

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role(
    GATEWAY_NAME, cred_provider_arn, secret_arn
)
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role["Role"]["Arn"])

### 2.3단계: AgentCore Gateway 생성

In [ ]:
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            gw_client_id
        ],  # Client는 Cognito에 구성된 ClientId와 반드시 일치해야 합니다
        "discoveryUrl": gw_cognito_discovery_url,
    }
}
create_response = agentcore_cp.create_gateway(
    name=GATEWAY_NAME,
    roleArn=agentcore_gateway_iam_role["Role"][
        "Arn"
    ],  # IAM 역할에는 Gateway 생성/목록 조회/가져오기/삭제 권한이 있어야 합니다
    protocolType="MCP",
    protocolConfiguration={
        "mcp": {"supportedVersions": ["2025-11-25"], "searchType": "SEMANTIC"}
    },
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway with MCP Server target",
)
# GatewayTarget 생성에 사용할 GatewayID 가져오기
gateway_id = create_response["gatewayId"]
gateway_url = create_response["gatewayUrl"]

print(gateway_id)

### 3단계(방법 1): MCP Server 대상 생성 중 암시적 동기화 

![암시적 동기화](./images/implicit.png)

1. 관리자 사용자가 MCP server endpoint, AgentCore Identity Credential Provider, return URL을 제공하여 `CreateGatewayTarget`을 호출합니다. 이를 통해 AgentCore Gateway에 연결할 MCP server와 OAuth 2.0 token을 가져오는 데 사용할 credential provider를 지정합니다. 동일한 flow가 `UpdateGatewayTarget` 및 `SynchronizeGatewayTargets` 작업에도 적용됩니다. 

2. AgentCore Gateway는 AgentCore Gateway workload identity와 `{gatewayId}{targetId}{uuid}` 형식의 사용자 ID를 전달하여 AgentCore Identity Credential Provider에 workload access token을 요청합니다. 이 workload access token은 이후 자격 증명 작업에서 AgentCore Gateway를 권한이 있는 호출자로 식별합니다. 

3. AgentCore Gateway는 workload access token을 사용하여 AgentCore Identity Credential Provider에 OAuth 2.0 access token을 요청합니다. 이때 관리자 사용자에게 권한 부여 URL과 session-URI가 제공됩니다. 이 단계에서 대상은 `Needs Authorization` 상태입니다.  

4. 관리자는 브라우저에서 권한 부여 URL을 열고 로그인한 다음 AgentCore Gateway에 요청된 권한을 부여합니다. 

5. 관리자가 동의하면 OAuth 2.0 권한 부여 서버가 AgentCore Identity Credential Provider에 등록된 callback endpoint로 authorization code를 보냅니다. 

6. Credential provider는 session URI와 함께 관리자 브라우저를 return URL로 리디렉션합니다. 관리자 애플리케이션은 사용자 ID와 2단계에서 반환된 session-URI를 제시하여 `CompleteResourceTokenAuth`를 호출합니다. Credential provider는 권한 부여 flow를 시작한 사용자(3단계)와 동의를 완료한 사용자가 동일한지 검증하여 권한 부여 URL이 실수로 공유되더라도 token 탈취를 방지합니다. AWS Console에서 flow를 시작한 경우 이 단계가 자동으로 처리됩니다. 다른 환경에서 시작한 경우 관리자가 `CompleteResourceTokenAuth` API를 직접 호출해야 합니다. 

7. 세션 바인딩 검증에 성공하면 credential provider가 OAuth 2.0 권한 부여 서버를 통해 authorization code를 OAuth 2.0 access token으로 교환합니다. 

8. 이 access token은 MCP server 대상의 도구 목록을 조회하는 데 사용되며, 대상에서 반환된 도구 정의는 AgentCore Gateway에 캐시됩니다.  

이후 대상을 업데이트하거나 동기화할 때는 access token을 재사용하지 않습니다. 대신 AgentCore Identity가 Authorization Server에서 새 access token을 가져옵니다.  

In [ ]:
CALLBACK_URL = "http://localhost:8080/callback"

target_response = agentcore_cp.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=f"github-mcp-server-{str(uuid.uuid4())[:4]}",
    description="Gateway with Github authorization code flow MCP server as target",
    targetConfiguration={
        "mcp": {"mcpServer": {"endpoint": "https://api.githubcopilot.com/mcp"}}
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cred_provider_arn,
                    "grantType": "AUTHORIZATION_CODE",
                    "defaultReturnUrl": CALLBACK_URL,
                    "scopes": ["repo", "user", "workflow"],
                }
            },
        }
    ],
)

target_id = target_response["targetId"]
authorizationUrl = target_response["authorizationData"]["oauth2"]["authorizationUrl"]
userId = target_response["authorizationData"]["oauth2"]["userId"]

print(f"Target ID: {target_id}")
print(f"Target status: {target_response['status']}")
print(f"User ID: {userId}")

대상을 생성하면 대상은 `Needs authorization` 상태가 됩니다. 이때 관리자 사용자는 AWS Console에서 직접 또는 권한 부여 URL로 직접 이동하여 권한 부여 요청을 완료해야 합니다. AWS Console에서 flow를 완료하면 세션 바인딩이 자동으로 처리됩니다. 다른 환경에서 시작한 경우 관리자가 `CompleteResourceTokenAuth` API를 직접 호출해야 합니다. 자세한 내용은 아래 코드 예제를 따르세요.  

![인증 필요](./images/need-auth.png)

In [ ]:
utils.start_callback_and_open_auth(authorizationUrl, "user-id", userId, REGION)

몇 초 후 대상 상태가 `Ready`, 권한 부여 상태가 `Authorized`로 표시됩니다.  

![암시적 동기화 완료](./images/complete-implicit.png)

### 3단계(방법 2): MCP Server 대상 생성 시 스키마를 미리 제공 

![스키마 사전 제공](./images/schema-upfront.png)

In [ ]:
with open("schema/github.json", "r") as f:
    tool_schema = f.read()

schema_target_response = agentcore_cp.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="github-mcp-server-schema-target",
    description="Github MCP Server with authorization code flow - inline schema",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": "https://api.githubcopilot.com/mcp",
                "mcpToolSchema": {"inlinePayload": tool_schema},
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cred_provider_arn,
                    "grantType": "AUTHORIZATION_CODE",
                    "defaultReturnUrl": CALLBACK_URL,
                    "scopes": ["repo", "user", "workflow"],
                }
            },
        }
    ],
)

schema_target_id = schema_target_response["targetId"]
print(f"Target ID: {schema_target_id}")
print(f"Target status: {schema_target_response['status']}")

이 경우 대상은 즉시 준비되며 권한 부여 상태는 `No authorization required`가 됩니다. 

![스키마 대상 생성 완료](./images/complete-schema.png)

### 4단계: AgentCore Gateway 호출

![AgentCore Gateway 호출](./images/invoke.png)

1. Gateway 사용자가 인바운드 권한 부여 token과 함께 `tools/list` 요청을 AgentCore Gateway로 보냅니다. 대상 생성 중 도구 정의가 캐시되었으므로 AgentCore Gateway는 캐시된 도구 정의를 즉시 반환합니다. 

2. Gateway 사용자가 인바운드 권한 부여 token과 함께 `tools/call` 요청을 AgentCore Gateway로 보냅니다. AgentCore Gateway가 이 사용자를 대신하여 MCP server를 호출하려면 access token이 필요하므로, 특정 MCP server 대상에 대한 OAuth authorization code flow가 시작됩니다. 

3. AgentCore Gateway는 workload identity와 인바운드 권한 부여 header의 사용자 JWT를 전달하여 AgentCore Identity에 workload access token을 요청합니다. 

4. AgentCore Gateway는 workload access token을 사용하여 credential provider에 OAuth 2.0 access token을 요청합니다. 아직 이 사용자에게 유효한 token이 없으므로 credential provider는 대신 권한 부여 URL과 session-URI를 반환합니다. 

6. AgentCore Gateway는 권한 부여 URL과 session URI를 Gateway 사용자에게 반환합니다. 사용자는 브라우저에서 권한 부여 URL을 열고 OAuth 2.0 권한 부여 서버에 로그인한 다음 요청된 권한을 부여합니다. AgentCore Gateway의 URL elicitation 응답 예시는 다음과 같습니다.  

    ```json 

    {     
        "jsonrpc": "2.0",                                                      
        "id": 3,     
        "error": {    
            "code": -32042,      
            "message": "This request requires more information.",    
            "data": { 
                "elicitations": [{ 
                "mode": "url", 
                "elicitationId": "<ID>",      
                "url": "<identity_url>/?request_uri=urn%3Aietf%3A...", 
                "message": "Please login to this URL for authorization." 
                }]       
            }        
        } 
    } 

    ``` 

6. 사용자가 동의하면 OAuth 2.0 권한 부여 서버가 AgentCore Identity Credential Provider에 등록된 callback endpoint로 authorization code를 보냅니다. 

7. Credential provider는 session URI와 함께 사용자의 브라우저를 return URL로 리디렉션합니다. 사용자 애플리케이션은 사용자의 JWT와 session-URI를 제시하여 `CompleteResourceTokenAuth`를 호출합니다. Credential provider는 권한 부여 flow를 시작한 사용자(4단계)와 동의를 완료한 사용자가 동일한지 검증하여 권한 부여 URL이 실수로 공유되더라도 token 탈취를 방지합니다. 

8. 세션 바인딩 검증에 성공하면 credential provider가 OAuth 2.0 권한 부여 서버를 통해 authorization code를 OAuth 2.0 access token으로 교환합니다. Credential provider는 workload identity와 사용자 identity에 연결된 Token Vault에 이 token을 캐시합니다. 

9. Gateway 사용자가 `tools/call` 요청을 다시 보내면 AgentCore Gateway는 workload identity와 사용자 identity를 사용하여 AgentCore Identity에서 캐시된 token을 가져오고, 이를 사용해 MCP server를 호출합니다. 

#### 4.1단계: 도구 목록 조회

In [ ]:
# Gateway 인바운드 인증을 위한 Cognito access token 가져오기
access_token = utils.get_token(
    gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION
)["access_token"]

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
    "Authorization": f"Bearer {access_token}",
    "Mcp-Protocol-Version": "2025-11-25",
}

# 1. 도구 목록 조회
list_response = requests.post(
    gateway_url,
    headers=headers,
    json={"jsonrpc": "2.0", "id": "list-tools", "method": "tools/list"},
)
print(json.dumps(list_response.json(), indent=2, default=str))

#### 4.2단계: 도구 호출

In [ ]:
# 2. 도구 호출
invoke_response = requests.post(
    gateway_url,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "id": "invoke-tool",
        "method": "tools/call",
        "params": {
            "name": "github-mcp-server-schema-target___search_repositories",
            "arguments": {"query": "amazon-bedrock-agentcore-samples", "perPage": 3},
        },
    },
)
print(json.dumps(invoke_response.json(), indent=2, default=str))

#### 4.3단계: Authorization Code flow 완료

In [ ]:
utils.complete_session_binding(invoke_response.json(), access_token, REGION)

#### 4.4단계: 도구 다시 호출

In [ ]:
# 세션 바인딩 후 도구 호출 재시도
invoke_response = requests.post(
    gateway_url,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "id": "invoke-tool-retry",
        "method": "tools/call",
        "params": {
            "name": "github-mcp-server-schema-target___search_repositories",
            "arguments": {"query": "amazon-bedrock-agentcore-samples", "perPage": 3},
        },
    },
)
print(json.dumps(invoke_response.json(), indent=2, default=str))

#### 4.5단계: 캐시된 자격 증명 재사용

In [ ]:
# 캐시된 자격 증명 재사용 - 이번에는 OAuth elicitation이 필요하지 않습니다
invoke_response = requests.post(
    gateway_url,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "id": "invoke-tool-cached",
        "method": "tools/call",
        "params": {
            "name": "github-mcp-server-schema-target___search_users",
            "arguments": {"query": "Eashan Kaushik", "perPage": 3},
        },
    },
)
print(json.dumps(invoke_response.json(), indent=2, default=str))

#### 4.6단계: 인증 강제 실행

인증 강제 실행은 캐시된 자격 증명을 사용하지 않고 새로운 OAuth authorization code flow를 시작합니다. 사용자의 권한이 변경되었거나 캐시된 token을 갱신해야 할 때 유용합니다. 도구 호출의 `_meta` 필드에 `forceAuthentication: true`를 전달하세요.

In [ ]:
# 인증 강제 실행 - 캐시된 자격 증명이 있어도 새로운 OAuth flow를 시작합니다
force_auth_response = requests.post(
    gateway_url,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "id": "invoke-tool-force-authentication",
        "method": "tools/call",
        "params": {
            "name": "github-mcp-server-schema-target___search_repositories",
            "arguments": {"query": "amazon-bedrock-agentcore-samples", "perPage": 3},
            "_meta": {
                "aws.bedrock-agentcore.gateway/credentialProviderConfiguration": {
                    "oauthCredentialProvider": {
                        "forceAuthentication": True,
                    }
                }
            },
        },
    },
)
print(json.dumps(force_auth_response.json(), indent=2, default=str))

utils.complete_session_binding(force_auth_response.json(), access_token, REGION)

In [ ]:
force_auth_response = requests.post(
    gateway_url,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "id": "invoke-tool-force-authentication",
        "method": "tools/call",
        "params": {
            "name": "github-mcp-server-schema-target___search_repositories",
            "arguments": {"query": "amazon-bedrock-agentcore-samples", "perPage": 3},
        },
    },
)
print(json.dumps(force_auth_response.json(), indent=2, default=str))

### 리소스 정리

In [ ]:
# Gateway 삭제(모든 대상도 함께 삭제됨)
utils.delete_gateway(agentcore_cp, gateway_id)

# Credential provider 삭제
agentcore_cp.delete_oauth2_credential_provider(name=CRED_PROVIDER_NAME)
print(f"Deleted credential provider: {CRED_PROVIDER_NAME}")


iam_client = boto3.client("iam")
role_name = f"agentcore-{GATEWAY_NAME}-role"
for policy_name in iam_client.list_role_policies(RoleName=role_name)["PolicyNames"]:
    iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
iam_client.delete_role(RoleName=role_name)
print(f"Deleted IAM role: {role_name}")

# Cognito domain을 먼저 삭제한 다음 user pool 삭제
pool_desc = cognito.describe_user_pool(UserPoolId=gw_user_pool_id)
domain = pool_desc["UserPool"].get("Domain")
if domain:
    cognito.delete_user_pool_domain(Domain=domain, UserPoolId=gw_user_pool_id)
    print(f"Deleted Cognito domain: {domain}")
cognito.delete_user_pool(UserPoolId=gw_user_pool_id)
print(f"Deleted Cognito user pool: {gw_user_pool_id}")